# Prescriptive layer — what the model recommends

In [31]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

ROOT  = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / 'data').exists())
DATA  = ROOT / 'data'
PLOTS = ROOT / 'results' / 'plots' / 'playbook'
PLOTS.mkdir(parents=True, exist_ok=True)

pb  = pd.read_parquet(DATA / 'adjustability_playbook.parquet')
val = pd.read_parquet(DATA / 'adjustability_value.parquet',
                      columns=['batter_id','batter_stand','label',
                               'runs_total','runs_count','runs_per_swing'])
gr  = pd.read_parquet(DATA / 'adjustability_gradients.parquet')

print(f'playbook:   {pb.shape}   axes: {list(pb.axis.unique())}   levers: {list(pb.lever.unique())}')
print(f'value:      {val.shape}')
print(f'gradients:  {gr.shape}   cells: {list(gr.cell.unique())}')

playbook:   (6745, 24)   axes: ['count', 'gamestate', 'platoon']   levers: ['swing_path_tilt', 'swing_length', 'vert_attack_angle', 'horz_attack_angle_pull', 'gradient']
value:      (466, 6)
gradients:  (18345, 13)   cells: ['0 strikes', '1 strike', '2 strikes', 'risp', 'empty', 'on1', 'opp-hand', 'same-hand']


## League consensus on the count axis

For each of the four levers, show: distribution of prescribed step_native across all movers (hitters
whose optimal step ≠ 0), with share-positive and median annotated. Ordered by mean held-out run gain.

In [32]:
LEVER_ORDER = ['vert_attack_angle', 'swing_path_tilt', 'swing_length', 'horz_attack_angle_pull']
LEVER_COLORS = {
    'vert_attack_angle':      '#C0392B',
    'swing_path_tilt':        '#D97B00',
    'swing_length':           '#1F6FB2',
    'horz_attack_angle_pull': '#5E3A8C',
}
# (pos_label, neg_label): pos_lbl used when median > 0, neg_lbl when median < 0
DIRECTION_LABEL = {
    'vert_attack_angle':      ('steepen', 'flatten'),
    'swing_path_tilt':        ('steepen', 'flatten'),
    'swing_length':           ('lengthen', 'shorten'),
    'horz_attack_angle_pull': ('pull',    'oppo'),
}

count_pb = pb[(pb.axis == 'count') & (pb.lever.isin(LEVER_ORDER))].copy()
movers   = count_pb[count_pb.step_sd != 0].copy()
n_total  = len(pb[(pb.axis == 'count') & (pb.lever == LEVER_ORDER[0])])

def hex_tint(hex_color, alpha=0.08):
    r = int(hex_color[1:3], 16)
    g = int(hex_color[3:5], 16)
    b = int(hex_color[5:7], 16)
    return (f'#{int(r*alpha+255*(1-alpha)):02x}'
            f'{int(g*alpha+255*(1-alpha)):02x}'
            f'{int(b*alpha+255*(1-alpha)):02x}')

rows = []
for lever in LEVER_ORDER:
    r        = movers[movers.lever == lever]
    med      = r['step_native'].median()
    pct_neg  = (r['step_sd'] < 0).mean() * 100
    pct_pos  = (r['step_sd'] > 0).mean() * 100
    cons_pct = pct_neg if med < 0 else pct_pos
    unit     = '"' if lever == 'swing_length' else '°'
    pos_lbl, neg_lbl = DIRECTION_LABEL[lever]
    direction = pos_lbl if med > 0 else neg_lbl
    gains    = r['runs_gain'].dropna()
    rows.append({
        'Metric':                 lever.replace('_', ' ').replace('horz attack angle pull', 'horiz. attack angle'),
        'Hitters who adjust':     f"{len(r)} / {n_total}  ({len(r)/n_total*100:.0f}%)",
        'Recommended change':     f"{direction}  ·  {cons_pct:.0f}% of movers",
        'Median step':            f"{med:+.1f}{unit}",
        'Avg gain (runs/season)': f"{gains.mean():+.2f}",
        '_color': LEVER_COLORS[lever],
        '_tint':  hex_tint(LEVER_COLORS[lever]),
        '_cons':  cons_pct,
        '_gain':  gains.mean(),
    })

tbl = pd.DataFrame(rows)

display_cols = ['Metric', 'Hitters who adjust', 'Recommended change',
                'Median step', 'Avg gain (runs/season)']
colors_list = tbl['_color'].tolist()
tints_list  = tbl['_tint'].tolist()
cons_list   = tbl['_cons'].tolist()
gain_list   = tbl['_gain'].tolist()

def style_fn(df):
    out = pd.DataFrame('', index=df.index, columns=df.columns)
    for i in df.index:
        tint  = tints_list[i]
        color = colors_list[i]
        gain  = gain_list[i]

        base = f'background-color: {tint}; color: #1a1a1a'
        for col in df.columns:
            out.loc[i, col] = base

        # Metric: brand color + bold
        out.loc[i, 'Metric'] = (
            f'background-color: {tint}; color: {color}; font-weight: bold')

        # Recommended change: brand color at 65% blend for legibility, bold
        blend = 0.65
        cr = int(int(color[1:3], 16) * blend + 255 * (1 - blend))
        cg = int(int(color[3:5], 16) * blend + 255 * (1 - blend))
        cb = int(int(color[5:7], 16) * blend + 255 * (1 - blend))
        cons_color = f'#{cr:02x}{cg:02x}{cb:02x}'
        out.loc[i, 'Recommended change'] = (
            f'background-color: {tint}; color: {cons_color}; font-weight: bold')

        # Avg gain: green / red + bold
        gcol = '#1a6b2e' if gain >= 0 else '#9b1c1c'
        out.loc[i, 'Avg gain (runs/season)'] = (
            f'background-color: {tint}; color: {gcol}; font-weight: bold')

    return out

styled = (
    tbl[display_cols].style
    .apply(style_fn, axis=None)
    .hide(axis='index')
    .set_caption('Two-strike prescription  ·  count axis  ·  466 units')
    .set_table_styles([
        {'selector': '',
         'props': [('border-collapse', 'collapse'), ('font-family', 'sans-serif'),
                   ('font-size', '13px'), ('background-color', 'white')]},
        {'selector': 'caption',
         'props': [('caption-side', 'top'), ('font-size', '14px'),
                   ('font-weight', 'bold'), ('color', '#1a1a1a'),
                   ('padding-bottom', '10px'), ('text-align', 'left')]},
        {'selector': 'thead tr th',
         'props': [('background-color', '#f0f0f0'), ('color', '#222'),
                   ('font-weight', 'bold'), ('border-bottom', '2px solid #bbb'),
                   ('padding', '10px 14px'), ('text-align', 'left')]},
        {'selector': 'tbody tr td',
         'props': [('padding', '9px 14px'), ('border-bottom', '1px solid #e0e0e0'),
                   ('vertical-align', 'middle')]},
        {'selector': 'tbody tr:last-child td',
         'props': [('border-bottom', '2px solid #bbb')]},
    ])
)

import dataframe_image as dfi
_out = PLOTS / 'count_axis_consensus_table.png'
dfi.export(styled, str(_out), table_conversion='matplotlib', dpi=150)
print(f'Saved → {_out}')
display(styled)


Saved → /Users/theoauyeung/Documents/swing_repertoire/results/plots/playbook/count_axis_consensus_table.png


Metric,Hitters who adjust,Recommended change,Median step,Avg gain (runs/season)
vert attack angle,321 / 466 (69%),flatten · 98% of movers,-5.3°,+3.45
swing path tilt,301 / 466 (65%),flatten · 90% of movers,-3.4°,+0.78
swing length,344 / 466 (74%),lengthen · 96% of movers,"+5.2""",+0.54
horiz. attack angle,327 / 466 (70%),pull · 67% of movers,+5.2°,+1.39


Top 5 and bottom 5 by runs with their count-axis prescription for each lever.
Shows that the model says something concrete and different for each hitter.

In [34]:
# Build a wide table: one row per hitter, one col per lever
count_wide = (
    pb[(pb.axis == 'count') & (pb.lever.isin(LEVER_ORDER))]
    .pivot_table(index=['batter_id','batter_stand'],
                 columns='lever',
                 values=['step_native','runs_gain'],
                 aggfunc='first')
)
count_wide.columns = [f'{stat}_{lev}' for stat, lev in count_wide.columns]
count_wide = count_wide.reset_index()

hitters = val.merge(count_wide, on=['batter_id','batter_stand'], how='inner')

top5    = hitters.nlargest(5, 'runs_total')
bot5    = hitters.nsmallest(5, 'runs_total')
display_df = pd.concat([top5, bot5]).reset_index(drop=True)

# Format for display
def fmt_step(row, lever):
    s = row[f'step_native_{lever}']
    if pd.isna(s) or s == 0:
        return 'no change'
    unit = '"' if lever == 'swing_length' else '°'
    return f'{s:+.1f}{unit}'

def fmt_gain(row, lever):
    g = row[f'runs_gain_{lever}']
    if pd.isna(g):
        return '—'
    return f'{g:+.2f}'

rows_out = []
for _, r in display_df.iterrows():
    rows_out.append({
        'Hitter':           r['label'],
        'runs_total':       f"{r['runs_total']:+.2f}",
        'VAA step':         fmt_step(r, 'vert_attack_angle'),
        'VAA gain':         fmt_gain(r, 'vert_attack_angle'),
        'Tilt step':        fmt_step(r, 'swing_path_tilt'),
        'Tilt gain':        fmt_gain(r, 'swing_path_tilt'),
        'Length step':      fmt_step(r, 'swing_length'),
        'Length gain':      fmt_gain(r, 'swing_length'),
        'HAA step':         fmt_step(r, 'horz_attack_angle_pull'),
        'HAA gain':         fmt_gain(r, 'horz_attack_angle_pull'),
    })

tbl = pd.DataFrame(rows_out)
display(tbl)

,Hitter,runs_total,VAA step,VAA gain,Tilt step,Tilt gain,Length step,Length gain,HAA step,HAA gain
0,José Caballero,+5.03,-5.3°,+4.87,-2.1°,+0.24,"+5.2""",+1.01,-8.6°,+2.37
1,Nolan Arenado,+4.68,-3.2°,+3.63,+3.4°,+0.44,"+7.8""",+1.67,no change,-0.04
2,Jose Altuve,+3.46,+7.9°,+0.88,-3.4°,+0.49,"+7.8""",+1.23,-5.2°,+2.36
3,Shohei Ohtani,+3.33,-3.2°,+5.11,-5.1°,+2.04,"+1.6""",+0.16,no change,+0.00
4,Josh Lowe,+3.09,-3.2°,+4.47,-2.1°,+0.08,"+3.1""",+0.66,+5.2°,+1.53
5,Jarren Duran,-2.80,-1.6°,+2.43,-2.1°,+0.94,"+5.2""",+1.30,+5.2°,+3.02
6,Brandon Marsh,-2.42,-3.2°,+2.93,-2.1°,+0.83,"+3.1""",+0.63,+5.2°,+2.26
7,Bo Naylor,-2.40,-5.3°,+4.13,-5.1°,+1.24,"+5.2""",+0.75,-5.2°,+0.67
8,Yandy Díaz,-2.29,-5.3°,+4.94,-2.1°,+0.17,"+7.8""",+0.60,+12.9°,+1.93
9,Josh Naylor,-2.25,-7.9°,+9.17,+1.0°,-0.05,"+5.2""",+0.97,-2.6°,-1.39


## Summary printout — the three concrete league-level numbers

These are the numbers that belong in the abstract / conclusion.

In [35]:
print('=== League-wide count-axis consensus (two-strike prescription) ===\n')
for lever in LEVER_ORDER:
    rows = movers[movers.lever == lever]
    n_total_units = len(count_pb[count_pb.lever == lever])
    n_mov  = len(rows)
    pct_mov = n_mov / n_total_units * 100
    pct_pos = (rows.step_sd > 0).mean() * 100
    pct_neg = 100 - pct_pos
    med     = rows.step_native.median()
    unit    = '"' if lever == 'swing_length' else '°'
    gains   = rows.runs_gain.dropna()
    mean_gain = gains.mean()

    direction = 'positive' if pct_pos >= 50 else 'negative'
    consensus_pct = max(pct_pos, pct_neg)

    print(f'{lever:<28}  {pct_mov:.0f}% of units move  '
          f'{consensus_pct:.0f}% {direction}  '
          f'median {med:+.1f}{unit}  '
          f'mean gain {mean_gain:+.3f} runs/season')

=== League-wide count-axis consensus (two-strike prescription) ===

vert_attack_angle             69% of units move  98% negative  median -5.3°  mean gain +3.445 runs/season
swing_path_tilt               65% of units move  90% negative  median -3.4°  mean gain +0.780 runs/season
swing_length                  74% of units move  96% positive  median +5.2"  mean gain +0.545 runs/season
horz_attack_angle_pull        70% of units move  67% positive  median +5.2°  mean gain +1.386 runs/season
